In [1]:
import numpy as np, pandas as pd
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.multitest import multipletests

df = pd.read_parquet("../data/interim/sparcs_2022_adults.parquet").reset_index(drop=True)
df["split"] = pd.read_parquet("../data/splits/split_seed0.parquet")["split"].values
df["dx_sys"] = df["ccsr_diagnosis_code"].str[:3]
cols = ["age_group", "gender", "race", "ethnicity", "type_of_admission",
        "emergency_department_indicator", "payment_typology_1", "dx_sys"]
X = OneHotEncoder(handle_unknown="ignore", min_frequency=50).fit_transform(df[cols].fillna("NA")).tocsr()

y = df["y"].values
h = df["permanent_facility_id"].values
tr = (df["split"] == "train").values
ce = (df["split"] == "cert").values
keep = df.groupby("permanent_facility_id")["y"].sum().loc[lambda s: s >= 100].index

def ll(p, y):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

fed = LogisticRegression(C=1.0, max_iter=300).fit(X[tr], y[tr])   # pooled model, stand-in for converged FedAvg

rows = []
for k in keep:
    mk = (h == k)
    a, b = mk & tr, mk & ce
    loc = LogisticRegression(C=1.0, max_iter=300).fit(X[a], y[a])
    l_loc = ll(loc.predict_proba(X[b])[:, 1], y[b])
    l_fed = ll(fed.predict_proba(X[b])[:, 1], y[b])
    d = l_fed - l_loc                      # positive = federated is worse
    rows.append((k, a.sum(), y[a].sum(), l_loc.mean(), l_fed.mean(), d.mean(), d.std(ddof=1) / np.sqrt(len(d))))

res = pd.DataFrame(rows, columns=["hospital", "n_train", "deaths_train", "loss_local", "loss_fed", "diff", "se"])
res["p_worse"] = 1 - stats.norm.cdf(res["diff"] / res["se"])
res["p_better"] = stats.norm.cdf(res["diff"] / res["se"])
res["sig_worse"] = multipletests(res["p_worse"], alpha=0.05, method="holm")[0]
res["sig_better"] = multipletests(res["p_better"], alpha=0.05, method="holm")[0]
res.to_csv("../outputs/tables/pilot_logreg_seed0.csv", index=False)

print(len(res), "hospitals")
print("raw share federated worse:", round((res["diff"] > 0).mean(), 3))
print("significantly worse (Holm):", res["sig_worse"].sum(), "=", round(res["sig_worse"].mean(), 3))
print("significantly better (Holm):", res["sig_better"].sum())
print("mean loss difference (fed - local):", round(res["diff"].mean(), 5))
res["size_q"] = pd.qcut(res["n_train"], 4, labels=["smallest", "q2", "q3", "largest"])
print(res.groupby("size_q", observed=True).agg(n=("diff", "size"), mean_diff=("diff", "mean"),
      raw_worse=("diff", lambda s: (s > 0).mean()), sig_worse=("sig_worse", "sum")))

126 hospitals
raw share federated worse: 0.675
significantly worse (Holm): 15 = 0.119
significantly better (Holm): 0
mean loss difference (fed - local): 0.00309
           n  mean_diff  raw_worse  sig_worse
size_q                                       
smallest  32   0.004893   0.656250          2
q2        31   0.001401   0.516129          1
q3        31   0.002854   0.741935          3
largest   32   0.003163   0.781250          9


In [2]:
grid = [0.25, 0.5, 0.75]
store = {a: [] for a in grid}
store["recal"] = []
for k in keep:
    mk = (h == k); a_, b_ = mk & tr, mk & ce
    loc = LogisticRegression(C=1.0, max_iter=300).fit(X[a_], y[a_])
    p_loc = loc.predict_proba(X[b_])[:, 1]
    z_b = fed.decision_function(X[b_]); p_fed = 1 / (1 + np.exp(-z_b))
    l_loc = ll(p_loc, y[b_])
    for a in grid:
        store[a].append(ll((1 - a) * p_loc + a * p_fed, y[b_]) - l_loc)
    rc = LogisticRegression(C=1e6, max_iter=300).fit(fed.decision_function(X[a_]).reshape(-1, 1), y[a_])
    store["recal"].append(ll(rc.predict_proba(z_b.reshape(-1, 1))[:, 1], y[b_]) - l_loc)

def summarize(ds):
    m = np.array([d.mean() for d in ds]); s = np.array([d.std(ddof=1) / np.sqrt(len(d)) for d in ds])
    z = m / s
    w = multipletests(1 - stats.norm.cdf(z), alpha=0.05, method="holm")[0]
    b = multipletests(stats.norm.cdf(z), alpha=0.05, method="holm")[0]
    return pd.Series({"mean_diff": m.mean(), "raw_worse": (m > 0).mean(), "sig_worse": int(w.sum()), "sig_better": int(b.sum())})

out = pd.DataFrame({("blend %.2f" % a): summarize(store[a]) for a in grid}).T
out.loc["pooled + recalibration"] = summarize(store["recal"])
print(out.round(5))

                        mean_diff  raw_worse  sig_worse  sig_better
blend 0.25               -0.00059    0.23016        0.0         2.0
blend 0.50               -0.00031    0.41270        1.0         1.0
blend 0.75                0.00075    0.57937        8.0         0.0
pooled + recalibration    0.00083    0.48413        3.0         0.0


In [3]:
rng = np.random.default_rng(0)
groups = ["gender", "age_band", "race_grp"]
delta = 0.01
cells = []
for k in keep:
    mk = (h == k); a_, b_ = mk & tr, mk & ce
    loc = LogisticRegression(C=1.0, max_iter=300).fit(X[a_], y[a_])
    p_loc = loc.predict_proba(X[b_])[:, 1]
    p_fed = fed.predict_proba(X[b_])[:, 1]
    yb = y[b_]; dk = df.loc[b_, groups].reset_index(drop=True)
    dk_tr = df.loc[a_, groups + ["y"]]
    for g in groups:
        for lev in dk[g].unique():
            if dk_tr[dk_tr[g] == lev]["y"].sum() < 20:
                continue
            idx = np.where(dk[g].values == lev)[0]
            e_loc = (p_loc[idx] - yb[idx]).mean(); e_fed = (p_fed[idx] - yb[idx]).mean()
            diff = abs(e_fed) - abs(e_loc)
            bs = rng.integers(0, len(idx), (200, len(idx)))
            d_b = np.abs((p_fed[idx][bs] - yb[idx][bs]).mean(1)) - np.abs((p_loc[idx][bs] - yb[idx][bs]).mean(1))
            cells.append((k, g, lev, len(idx), diff, np.percentile(d_b, 2.5)))

cal = pd.DataFrame(cells, columns=["hospital", "var", "group", "n", "diff", "lo95"])
cal.to_csv("../outputs/tables/pilot_calibration_seed0.csv", index=False)
print(len(cal), "testable hospital-group cells in", cal["hospital"].nunique(), "hospitals")
print("raw: pooled calibration worse than local by > 1 pt:", round((cal["diff"] > delta).mean(), 3))
print("lower bound > 1 pt:", int((cal["lo95"] > delta).sum()), "cells in", cal.loc[cal["lo95"] > delta, "hospital"].nunique(), "hospitals")
print(cal.groupby("var").agg(cells=("diff", "size"), raw_worse=("diff", lambda s: (s > delta).mean()),
      sig=("lo95", lambda s: int((s > delta).sum()))))

768 testable hospital-group cells in 126 hospitals
raw: pooled calibration worse than local by > 1 pt: 0.266
lower bound > 1 pt: 30 cells in 18 hospitals
          cells  raw_worse  sig
var                            
age_band    260   0.303846   10
gender      252   0.234127   14
race_grp    256   0.257812    6


In [5]:
rng = np.random.default_rng(1)
cands = {"pooled": 1.0, "blend 0.75": 0.75, "blend 0.50": 0.5, "blend 0.25": 0.25}
rows = []
for k in keep:
    mk = (h == k); a_, b_ = mk & tr, mk & ce
    loc = LogisticRegression(C=1.0, max_iter=300).fit(X[a_], y[a_])
    p_loc = loc.predict_proba(X[b_])[:, 1]; p_fed = fed.predict_proba(X[b_])[:, 1]
    yb = y[b_]; dk = df.loc[b_, groups].reset_index(drop=True); dk_tr = df.loc[a_, groups + ["y"]]
    for g in groups:
        for lev in dk[g].unique():
            if dk_tr[dk_tr[g] == lev]["y"].sum() < 20: continue
            idx = np.where(dk[g].values == lev)[0]; bs = rng.integers(0, len(idx), (300, len(idx)))
            for name, a in cands.items():
                p_c = (1 - a) * p_loc + a * p_fed
                d_obs = abs((p_c[idx] - yb[idx]).mean()) - abs((p_loc[idx] - yb[idx]).mean())
                d_b = np.abs((p_c[idx][bs] - yb[idx][bs]).mean(1)) - np.abs((p_loc[idx][bs] - yb[idx][bs]).mean(1))
                se = d_b.std(ddof=1)
                p = 1 - stats.norm.cdf((d_obs - delta) / se) if se > 0 else 1.0   # H0: harm <= delta
                rows.append((name, k, g, lev, p))
cc = pd.DataFrame(rows, columns=["cand", "hospital", "var", "group", "p"])

def loss_set(ds):
    m = np.array([d.mean() for d in ds]); s = np.array([d.std(ddof=1) / np.sqrt(len(d)) for d in ds])
    w = multipletests(1 - stats.norm.cdf(m / s), alpha=0.05, method="holm")[0]
    return set(keep[w])

loss_sets = {"pooled": set(res.loc[res["sig_worse"], "hospital"]),
             "blend 0.75": loss_set(store[0.75]), "blend 0.50": loss_set(store[0.5]), "blend 0.25": loss_set(store[0.25])}
out = []
for name in cands:
    t = cc[cc["cand"] == name].copy()
    t["sig"] = multipletests(t["p"], alpha=0.05, method="holm")[0]
    cal_h = set(t.loc[t["sig"], "hospital"])
    out.append((name, int(t["sig"].sum()), len(cal_h), len(loss_sets[name]), len(cal_h | loss_sets[name])))
print(pd.DataFrame(out, columns=["candidate", "cal_cells_harmed", "hosp_cal", "hosp_loss", "hosp_loss_or_cal"]))

    candidate  cal_cells_harmed  hosp_cal  hosp_loss  hosp_loss_or_cal
0      pooled                16        11         15                18
1  blend 0.75                 4         4          8                11
2  blend 0.50                 0         0          1                 1
3  blend 0.25                 0         0          0                 0
